# Algorithmic Methods of Data Mining and Laboratory

# Lab Session 04 - Algorithm Performance Measures

The performance of an algorithm can be evaluated using two complementary measures:
1. **Operation Counts:** A machine-independent deterministic counter that keeps track of the number of "basic operations" performed by an algorithm.
    - This measure is often used to define asymptotic bounds for an algorithm.
2. **Wall-Clock Time:** The experienced running time, which, however, is noisy and dependent on the machine and its overhead.endent on the machine and on Python overheads (a built-in such as sorted is written in C and will look much faster than its operation count suggests).

In [ ]:
import operator
import time
from collections import defaultdict

In [ ]:
class OpCounter:
  # This class counts named operations,
  def __init__(self):
    self.counts = defaultdict(int)

  def inc(self, name="ops", k=1):
    self.counts[name] += k

  def reset(self):
    self.counts.clear()

  def total(self):
    return sum(self.counts.values())

  def __repr__(self):
    return "OpCounter(" + str(dict(self.counts)) + ")"


OPS = OpCounter()

In [ ]:
class Counted:
  # This class counts the number of operations and comparisons performed on a (numeric) object.
  __slots__ = ("v",)

  def __init__(self, v):
    self.v = v

  def __repr__(self):
    return f"Counted({self.v!r})"

  def __hash__(self):
    return hash(self.v)

  def __neg__(self):
    OPS.inc("arith")
    return Counted(-self.v)


def _val(x):
  return x.v if isinstance(x, Counted) else x


def _install(cls):
  for name, op in [("lt", operator.lt), ("le", operator.le), ("gt", operator.gt),
                    ("ge", operator.ge), ("eq", operator.eq), ("ne", operator.ne)]:
    def cmp(self, other, op=op):
      OPS.inc("cmp")
      return op(self.v, _val(other))
    setattr(cls, f"__{name}__", cmp)

  for name, op in [("add", operator.add), ("sub", operator.sub),
                    ("mul", operator.mul), ("floordiv", operator.floordiv),
                    ("mod", operator.mod)]:
    def fwd(self, other, op=op):
      OPS.inc("arith")
      return Counted(op(self.v, _val(other)))

    def rev(self, other, op=op):
      OPS.inc("arith")
      return Counted(op(_val(other), self.v))
    setattr(cls, f"__{name}__", fwd)
    setattr(cls, f"__r{name}__", rev)

_install(Counted)

In [ ]:
# Convert items into Counted objects.
def counted(a):
  return [Counted(x) for x in a]

def uncount(a):
  return [x.v for x in a]

def measure(f, *args, repeat=1, **kwargs):
  # This function runs f(*args, *kwargs) for repeat times.
  # Along with the result, it returns the best running time and the number of operations performed in the last run.
  times, result = [], None
  OPS_counts = defaultdict(int)
  for _ in range(repeat):
    OPS.reset()
    t0 = time.perf_counter()
    result = f(*args, **kwargs)
    times.append(time.perf_counter() - t0)
    OPS_counts = {k: OPS_counts[k] + v for k, v in OPS.counts.items()}
  times.sort()
  OPS_counts = {k: v/repeat for k, v in OPS_counts.items()}
  return result, times, OPS_counts


def growth_table(f, make_input, sizes, repeat=3):
  # This function runs f on increasingly large inputs and prints the time, operations and performance ratio between consecutive runs.
  print(f"{'n':>10} {'time (s)':>12} {'ops':>14} {'time ratio':>11} {'ops ratio':>10}")
  prev_t = prev_o = None
  for n in sizes:
    _, t, ops = measure(f, *make_input(n), repeat=repeat)
    o = sum(ops.values())
    rt = f"{t / prev_t:.2f}" if prev_t else "-"
    ro = f"{o / prev_o:.2f}" if prev_o and o else "-"
    print(f"{n:>10} {t:>12.6f} {o:>14} {rt:>11} {ro:>10}")
    prev_t, prev_o = t, o

In [ ]:
def YOUR_ALGORITHM(a):
  b = [a + 1 for a in sorted(a)]
  return b
a = counted([0, -2, 1, -3, 4, -1, 2, 1, -5, 4])

result, t, ops = measure(YOUR_ALGORITHM, a, repeat=10)
result = uncount(result)
print("Result:", result)
print("Times:", t)
print("Operations:", ops)

## Exercise 1 - Maximum Subarray

Given a list of numbers, find a contiguous non-empty subarray ```a[i, ..., j]``` whose sum is maximum.

In [ ]:
def max_subarray(a):
  if not a:
    return (0, -1, -1)

  best_sum = float('-inf')
  best_i, best_j = -1, -1

  for i in range(len(a)):
    current_sum = 0
    for j in range(i, len(a)):
      current_sum += a[j]
      if current_sum > best_sum:
        best_sum = current_sum
        best_i = i
        best_j = j

  return (best_sum, best_i, best_j)

# Quick test.
max_subarray([-2, 1, -3, 4, -1, 2, 1, -5, 4])
result, t, ops = measure(max_subarray,[-2, 1, -3, 4, -1, 2, 1, -5, 4], repeat=10)   # ops = {'arith': ..., 'cmp': ...}
# result = uncount(result)
print("Result", result)
print("Times:", t)
print("Operations:", ops)

## Exercise 2 - Merging Sorted Lists

Given $k$ lists, each sorted in increasing order, produce a single sorted list containing all their elements.

In [ ]:
def merge_k(lists):
  result=[]
  for l in lists:
    if len(l) > 0:
      for char in l:
        result.append(char)
  result.sort()

  return result

# Quick test.
result, t, ops = measure(merge_k,[[1, 4, 7], [2, 5], [], [3, 6, 8, 9]], repeat=10)
# result = uncount(result)
print("Result", result)
print("Times:", t)
print("Operations:", ops)

## Exercise 3 - Reservoir Sampling

You receive a stream of items that you can read only once.

Assuming you do not know the length of the stream in advance, you must return a sample of $k$ items such that every item of the stream has the same probability of being in the sample.

Note that you can only use $O(k)$ memory.

In [ ]:
import random

def make_stream(n):
  for x in range(n):
    # Implement the stream as a function generator.
    yield x

# Try a simple, but inappropriate, version of the sampling method.
def bad_sample(stream, k, rng):
  k = 1
  result = None
  for element_number, element in enumerate(stream):
    if random.uniform(0,1) < 1 / (element_number +1):
      result = element
  return result
bad_sample(make_stream(6), 1, 12)

# Implement reservoir sample.
def reservoir_sample(stream, k, rng):
  result= []
  for element_number, element in enumerate(stream):
    if random.uniform(0,1) < 1 / (element_number +1):
      if len(result)<k:
        result.append(element)
      else:
        result[random.randint(0, k - 1)] = element
  return result
bad_sample(make_stream(6), 5, 56)

# Quick test using a stream.
def make_stream(n):
  for x in range(n):
    yield x
make_stream(5)

## Exercise 4 - Bloom Filtering

A Bloom filter is a data structure that lets a program check whether it has already seen a certain item.

Generally speaking, this data structure is memory efficient, although it tends to be make (one-sided) mistakes:
- If a value has been inserted, the answer is always "Yes".
  - This implies that there is no risk of false negative results.
- If a value has not been inserted, the answer is typically "No", although a "Yes" might be returned with small probability.
  - This implies that there might be the risk of false positive results.

More specifically, false positives in a Bloom filter occur whenever the chosen hash function causes a collision, meaning that $x \neq y$ but $h(x) = h(y)$.